# MetaPredictor on the GLORYx-37 set (for the MetaBench comparison)

Runtime > GPU (T4 is fine). Run all cells. The last cell downloads `metapredictor_gloryx.json`
(and the raw `predict.csv` as a fallback) — send both back. ~10-20 min on GPU.

In [ ]:
# 1) Clone MetaPredictor (weights are committed in-repo, ~1.8 GB) + install OpenNMT-py.
!git clone --depth 1 https://github.com/zhukeyun/Meta-Predictor.git /content/mp
%cd /content/mp
!pip -q install OpenNMT-py==2.3.0 rdkit 2>&1 | tail -2
import onmt, torch; print('onmt', onmt.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2) The predict scripts reference model/SoM_identifier and model/metabolite_predictor, but the
#    committed dirs have spaces. Rename to underscores so the scripts find the weights.
import os
for a, b in [('model/SoM identifier', 'model/SoM_identifier'), ('model/metabolite predictor', 'model/metabolite_predictor')]:
    if os.path.isdir(a) and not os.path.isdir(b):
        os.rename(a, b)
print(sorted(os.listdir('model')))

In [ ]:
# 3) Fetch the GLORYx-37 parents from GitHub (escape the stray SMILES backslashes), write input.csv.
import urllib.request, json, re
url = 'https://raw.githubusercontent.com/christinadebruynkops/GLORYx/master/datasets/test_dataset/gloryx_test_dataset.json'
raw = urllib.request.urlopen(url).read().decode()
data = json.loads(re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', raw))
parents = [p['smiles'] for p in data if p.get('smiles')]
with open('input.csv', 'w') as f:
    for i, smi in enumerate(parents):
        f.write(f'{i},{smi}\n')
print(len(parents), 'parents written to input.csv')

In [ ]:
# 4) Run the official two-stage pipeline (SoM identifier -> metabolite predictor), top-15.
!python prepare_input_file.py -input_file input.csv -output_file processed.txt
!mkdir -p prediction
!bash predict-top15.sh processed.txt ./prediction input.csv
print('--- prediction/ ---'); import os; print(os.listdir('prediction'))

In [ ]:
# 5) Inspect the output, build {parent_smiles: [metabolite_smiles]} JSON, download it + the raw csv.
import pandas as pd, json
df = pd.read_csv('prediction/predict.csv')
print('predict.csv columns:', list(df.columns)); print(df.head(3).to_string())
# Heuristic mapping: a column holding the input/substrate SMILES + columns holding predicted metabolites.
cols = [c for c in df.columns]
sub_col = next((c for c in cols if 'substrate' in c.lower() or 'input' in c.lower() or c.lower()=='smiles'), cols[0])
met_cols = [c for c in cols if c != sub_col and ('metab' in c.lower() or 'predict' in c.lower() or 'top' in c.lower())]
if not met_cols:
    met_cols = [c for c in cols if c != sub_col]
preds = {}
for _, row in df.iterrows():
    sub = str(row[sub_col])
    mets = [str(row[c]) for c in met_cols if isinstance(row[c], str) and row[c] and str(row[c]) != 'nan']
    preds.setdefault(sub, [])
    for m in mets:
        if m not in preds[sub]:
            preds[sub].append(m)
json.dump(preds, open('metapredictor_gloryx.json', 'w'), indent=2)
print('wrote', len(preds), 'parents,', sum(len(v) for v in preds.values()), 'metabolites')
from google.colab import files
files.download('metapredictor_gloryx.json')
files.download('prediction/predict.csv')  # raw fallback in case the mapping above needs fixing